# Installing packages and loading data file

In [2]:
# Install packages 
#%pip install pandas numpy matplotlib scikit-learn ydata-profiling

In [5]:
# Import relavent packages 
import joblib
import pandas as pd
import numpy as np

In [11]:
# Load practice test dataset 
df_prac = pd.read_csv("/Users/muyuanli/Library/CloudStorage/OneDrive-CornellUniversity/Desktop/ANSC 6040/PracticeTest.csv")

In [7]:
df_prac.head()

,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking,MilkingId
0,NaN,NaN,NaN,NaN,2021-10-22,2.812273,2.998246,5.923916,10.387265,222,1,8
1,NaN,NaN,NaN,NaN,2021-04-08,3.401943,3.701314,6.899140,12.836664,223,2,17
2,NaN,NaN,NaN,NaN,2021-09-08,3.810176,4.998588,8.464034,8.754333,135,1,19
3,NaN,NaN,NaN,NaN,2021-05-22,2.494758,2.998246,5.347854,7.393556,174,3,27
4,NaN,NaN,NaN,NaN,2020-11-04,2.721554,2.299713,5.574650,8.300740,184,1,37


In [9]:
df_prac = df_prac.rename(columns={'Avgmilkflow': 'AvgMilkFlow'})
df_prac = df_prac.rename(columns={'milking': 'Milking'})
df_prac = df_prac.rename(columns={'DurationSession_sec': 'DurationSession_Sec'})

In [10]:
# Load Model
model_rf = joblib.load("Project1FinalModel_ML.pkl")
features_trimmed = joblib.load("Feature_list.pkl")
X_new = df_prac[features_trimmed]

In [12]:
#Predict in batches, writing directly to disk to save memory
batch_size = 10000
first = True

for start in range(0, len(X_new), batch_size):
    end = start + batch_size
    batch = X_new.iloc[start:end]
    batch_pred = model_rf.predict(batch)

    batch_results = pd.DataFrame({
        "MilkingId": df_prac["MilkingId"].iloc[start:end].values,
        "predicted_cow_id": batch_pred
    })

    batch_results.to_csv("predicted_cow_ids.csv", mode="a", header=first, index=False)
    first = False

    if start % 500000 == 0:
        print(f"Processed {start} of {len(X_new)}")

print("Done.")

Processed 0 of 1701003
Processed 500000 of 1701003
Processed 1000000 of 1701003
Processed 1500000 of 1701003
Done.


In [13]:
df_pred = pd.read_csv("/Users/muyuanli/Library/CloudStorage/OneDrive-CornellUniversity/Desktop/ANSC 6040/ANSC-6040-Project-1/predicted_cow_ids.csv")

In [20]:
df_pred.head(20)

,MilkingId,predicted_cow_id
0,8,-5632035610966841344
1,17,-8773657057901642752
2,19,5959126385833853952
3,27,-2916298632258562560
4,37,1953030405829834752
5,39,-6765949298022812672
6,49,-1137436016399384448
7,51,-484093260715573888
8,59,1855817576332041984
9,64,2985539753013079552


In [15]:
df_pred.shape

(1701003, 2)

In [19]:
df_pred['predicted_cow_id'].nunique()

420